# Optymalizacja hiperparametrów - Ray Tune

Ten notebook przeprowadza automatyczne przeszukiwanie przestrzeni hiperparametrów
za pomocą biblioteki **Ray Tune** w celu znalezienia optymalnej konfiguracji
dla modelu DeepMReye trenowanego na danych resting-state.

Przeszukiwana przestrzeń obejmuje:
- learning rate (`lr`)
- delta funkcji Smooth L1 (`smooth_l1_delta`)
- poziom szumu gaussowskiego (`gaussian_noise`)
- współczynnik dropout (`dropout_rate`)
- wagę straty confidence (`loss_confidence`)

Przeprowadzono **30 prób**. Najlepsza konfiguracja jest następnie używana do treningu finalnego modelu.

In [ ]:
import os 
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async" 
import tensorflow as tf 
tf.keras.backend.clear_session() 
import pandas as pd 
from deepmreye import analyse, architecture, preprocess, train 
from deepmreye.util import data_generator, model_opts, util 
import numpy as np 
from smooth_loss import train_model_with_smoothl1

gpus = tf.config.experimental.list_physical_devices('GPU') 
tf.config.experimental.set_memory_growth(gpus[0], True)


## Funkcje pomocnicze

`create_holdout_generators` opakowuje generator danych DeepMReye.
Przyjmuje jawne listy ścieżek `train_list` / `test_list` i zwraca generatory
treningowe, testowe oraz per-podmiot używane podczas ewaluacji.

In [ ]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )

## Domyślna konfiguracja modelu

Wczytywane są domyślne hiperparametry DeepMReye.
Część z nich zostanie nadpisana przez Ray Tune podczas przeszukiwania.

In [ ]:
opts = model_opts.get_opts()
print(opts)

## Wczytywanie plików resting-state

Zbierane są ścieżki do plików `.npz` wszystkich 29 podmiotów ze zbioru resting-state.

In [ ]:
all_files=[]
for root, dirs, files in os.walk('/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/'):
    for name in files:
        all_files.append(os.path.join(root,name))
        print(os.path.join(root,name))

In [ ]:
datasets = {}
for f in all_files:
    dataset_name = os.path.basename(os.path.dirname(f))
    datasets.setdefault(dataset_name, []).append(f)

train_list_rs, test_list_rs = [], []
for dataset_name, files in datasets.items():
    files = sorted(files)                
    np.random.shuffle(files)             
    split_idx = int(0.8 * len(files))   
    train_list_rs.extend(files[:split_idx])
    test_list_rs.extend(files[split_idx:])

## Podział na zbiory treningowy, walidacyjny i testowy

Stały podział wczytywany z pliku `.txt` zapewnia reprodukowalność eksperymentu.

- `train_list_rs` - 18 podmiotów (właściwy trening w każdej próbie Ray Tune)
- `validation_list` - 5 podmiotów (walidacja podczas przeszukiwania)
- `test_list` - 6 podmiotów (wydzielony zbiór testowy, nieużywany podczas tuningu)

In [ ]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt',dtype=str)

train_list = train_list_rs[5:]
validation_list = train_list_rs[:5]
test_list = test_list_rs
train_list_rs = train_list

In [ ]:
print("Total train:", len(train_list_rs))
print("Fine tunning total:", len(validation_list))
print("Total test:", len(test_list))

## Konwersja ścieżek i tworzenie generatorów danych

Ścieżki względne konwertowane są na bezwzględne.
Generatory są tworzone z augmentacją danych (obroty 3D, przesunięcia, zoom)
i `mixed_batches=True` dla stabilności treningu.

In [ ]:
train_list_rs_absolute = []
for f in train_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    train_list_rs_absolute.append(absolute_path)

test_list_rs_absolute = []
for f in test_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    test_list_rs_absolute.append(absolute_path)

In [ ]:
generators_rs = create_holdout_generators(
    train_list=train_list_rs_absolute,
    test_list=test_list_rs_absolute, 
    batch_size=opts['batch_size'],
    augment_list=(
        (opts['rotation_x'], opts['rotation_y'], opts['rotation_z']), 
        opts['shift'], 
        opts['zoom']
    ),
    mixed_batches=True
)

## Inicjalizacja Ray Tune

Importowane są biblioteki Ray i konfigurowany jest lokalny klaster obliczeniowy.
Wyłączone są metryki i dashboard Ray, aby zminimalizować zużycie zasobów.
Katalog `ray_spill` służy do zrzutu obiektów gdy pamięć RAM jest pełna.

In [ ]:
import json
import ray

from ray import tune
from ray.train import report
from ray.tune.schedulers import ASHAScheduler

from smooth_loss import train_model_with_smoothl1


In [ ]:
!rm -rf /tmp/ray/*
!rm -rf ./ray_results/*
!rm -rf ./ray_spill/*

## Konfiguracja środowiska Ray i funkcja treningowa

Ray jest inicjalizowany z konfiguracją spill na dysk.

Funkcja `train_deepmreye_ray` definiuje jedną **próbę** przeszukiwania:
1. Kopiuje bazowe opcje i nadpisuje je konfiguracją próby
2. Tworzy generatory dla 18 podmiotów treningowych i 5 walidacyjnych
3. Trenuje model przez 50 epok (po 10 kroków każda)
4. Raportuje najlepszą stratę walidacyjną do Ray Tune

`freeze_backbone=False` - szkielet CNN jest trenowany od zera (brak transfer learning).

In [ ]:
opts["epochs"] = 50
opts["steps_per_epoch"] = 10
opts["validation_steps"] = 4

results_path = "/mnt/compneuro/deepmreye_finetuning/Zosia/ray_results"
spill_path = "/mnt/compneuro/deepmreye_finetuning/Zosia/ray_spill"

if ray.is_initialized():
    ray.shutdown()

os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_DISABLE_DASHBOARD"] = "1"

ray.init(
    _system_config={
        "object_spilling_config": json.dumps(
            {"type": "filesystem", "params": {"directory_path": spill_path}}
        )
    },
    ignore_reinit_error=True
)


## Przestrzeń przeszukiwania hiperparametrów

Ray Tune losowo próbkuje kombinacje parametrów z poniższej przestrzeni:

| Parametr | Zakres / Wartości |
|---|---|
| `lr` | loguniform(1e-6, 5e-5) |
| `smooth_l1_delta` | {0.03, 0.05, 0.1} |
| `gaussian_noise` | {0.0, 0.01, 0.02} |
| `dropout_rate` | {0.0, 0.1, 0.2} |
| `loss_confidence` | {0.25, 0.5, 1.0} |


In [ ]:
train_list = train_list_rs[5:]       
val_list   = train_list_rs[:5]      
test_list  = test_list_rs            

base_opts = opts.copy()
def train_deepmreye_ray(config):
    print("Trial started", config, flush=True)

    try:
        trial_opts = base_opts.copy()
        trial_opts.update(config)

        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            tf.config.experimental.set_memory_growth(gpus[0], True)

        generators = data_generator.create_generators(
            full_training_list=train_list,   
            full_testing_list=val_list,     
            batch_size=trial_opts["batch_size"]
        )

        generators = (
            *generators,
            train_list,
            val_list
        )

        model, model_inf, history = train_model_with_smoothl1(
            dataset="ray_trial",
            generators=generators,
            opts=trial_opts,
            is_resting_state=True,
            save=False,
            verbose=0,
            use_multiprocessing=False,
            workers=1,
            freeze_backbone=False   
        )

        val_losses = history.history.get("val_smoothl1_loss", [])
        best_loss = min(val_losses)

        tune.report({"loss": best_loss})

    except Exception:
        import traceback
        traceback.print_exc()
        tune.report({"loss": float("inf")})


## Uruchomienie przeszukiwania (30 prób)

Ray Tune uruchamia 30 niezależnych prób, każda z losową konfiguracją z przestrzeni powyżej.
Optymalizowany jest **minimalny val loss**.
Po zakończeniu wyświetlana jest najlepsza znaleziona konfiguracja.

In [ ]:
search_space = {
    "lr": tune.loguniform(1e-6, 5e-5),
    "smooth_l1_delta": tune.choice([0.03, 0.05, 0.1]),
    "gaussian_noise": tune.choice([0.0, 0.01, 0.02]),
    "dropout_rate": tune.choice([0.0, 0.1, 0.2]),
    "loss_confidence": tune.choice([0.25, 0.5, 1.0]),
    "batch_size": 8
}

## Najlepsza konfiguracja - finalne opcje modelu

Najlepsza konfiguracja znaleziona przez Ray Tune:
- `lr` = 9.85e-06
- `smooth_l1_delta` = 0.03
- `gaussian_noise` = 0.01
- `dropout_rate` = 0.1
- `loss_confidence` = 0.5

Opcje są kopiowane do `final_opts` z pełną liczbą epok (100) i kroków (60)
do finalnego treningu na całym zbiorze treningowym.

In [ ]:
analysis = tune.run(
    train_deepmreye_ray,
    config=search_space,
    metric="loss",
    mode="min",
    num_samples=30,
    raise_on_failed_trial=False,
    resources_per_trial={"cpu": 4, "gpu": 1}
)

print("Best config:", analysis.best_config)


In [ ]:
final_opts = opts.copy()

final_opts.update({
    "lr": 9.854681852379987e-06,
    "smooth_l1_delta": 0.03,
    "gaussian_noise": 0.01,
    "dropout_rate": 0.1,
    "loss_confidence": 0.5,
    "batch_size": 8,
})

final_opts["epochs"] = 100
final_opts["steps_per_epoch"] = 60
final_opts["validation_steps"] = 15

## Trening modelu z najlepszymi hiperparametrami

Po zakończeniu przeszukiwania przestrzeni parametrów przez Ray Tune, model jest trenowany
na pełnym zbiorze treningowym (18 podmiotów) z użyciem najlepszej znalezionej konfiguracji.


In [ ]:
model_best, model_inference_best, history_best = train_model_with_smoothl1(
    dataset="resting_state_best_config",
    generators=generators_rs,
    opts=final_opts,
    is_resting_state=True,
    save=True,
    verbose=1
)

## Ewaluacja na zbiorze treningowym

Sprawdzenie dopasowania modelu do danych treningowych (18 podmiotów).
Wysoka korelacja Pearsona r przy stosunkowo niskim R² oznacza prawidłowe dopasowanie bez nadmiernego przeskalowania amplitudy.

Wyniki: `figures\3-Train_hyperparameter_tunning.png`

In [ ]:
generators_train_eval = data_generator.create_generators(
    train_list_rs_absolute,
    train_list_rs_absolute
)
generators_train_eval = (
    *generators_train_eval,
    train_list_rs_absolute,
    train_list_rs_absolute
)

In [ ]:
(evaluation_train, scores_train) = train.evaluate_model(
    dataset='resting_state_train',
    model=model_inference_best,
    generators=generators_train_eval,
    save=True,
    model_description='best_config_train',
    verbose=2
)

In [ ]:
fig = analyse.visualise_predictions_slider(
    evaluation_train,
    scores_train,
    color="rgb(0, 150, 175)",
    bg_color="rgb(255,255,255)",
    ylim=[-2, 2]
)
fig.show()

## Ewaluacja na zbiorze testowym

Ewaluacja na 6 wydzielonych podmiotach testowych - rzeczywista miara generalizacji modelu.

In [ ]:
generators_test_eval = data_generator.create_generators(
    test_list_rs_absolute,
    test_list_rs_absolute
)
generators_test_eval = (
    *generators_test_eval,
    test_list_rs_absolute,
    test_list_rs_absolute
)

In [ ]:
(evaluation_test, scores_test) = train.evaluate_model(
    dataset='resting_state_test',
    model=model_inference_best,
    generators=generators_test_eval,
    save=True,
    model_description='best_config_test',
    verbose=2
)

In [ ]:
fig = analyse.visualise_predictions_slider(
    evaluation_test,
    scores_test,
    color="rgb(0, 150, 175)",
    bg_color="rgb(255,255,255)",
    ylim=[-2, 2]
)
fig.show()